In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import KFold

import tensorflow as tf

import keras
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Flatten, MaxPool2D, Conv2D
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import L1L2, L1, L2
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from tqdm import tqdm
import gc

2024-08-04 23:05:45.734194: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-04 23:05:45.734316: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-04 23:05:45.882274: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
negative_pairs_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/txt_interac/negative_pairs.txt"
positive_pairs_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt"

positive_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(positive_pairs_path,"r").readlines()]
negative_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(negative_pairs_path,"r").readlines()]

all_pairs = positive_pairs + negative_pairs

data = np.load("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Red Convolucional/data2D.npy")
labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)
labels = to_categorical(labels, num_classes=2)

In [3]:
def get_model(dp,n1,n2,n3):
    inx = Input(shape=(bin_size*3, bin_size, 6))
    x = BatchNormalization()(inx)
    x = Conv2D(n1, kernel_size=(3, 3), activation="relu", kernel_regularizer=L1(0.002))(x)
    #x = Dropout(0.3)(x) antes de dp estaba 0.3
    x = Dropout(dp)(x)
    x = Conv2D(n1, kernel_size=(3, 3), activation="relu", kernel_regularizer=L1(0.002))(x)
    x = Conv2D(n2, kernel_size=(3, 3), activation="relu", kernel_regularizer=L1(0.002))(x)
    x = Conv2D(n2, kernel_size=(3, 3), activation="relu")(x)
    x = Dropout(dp)(x)
    x = Conv2D(n3, kernel_size=(3, 3), activation="relu")(x)
    x = MaxPool2D(pool_size=(2,2))(x)
    x = Flatten()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dense(128, activation="relu")(x)
    x = Dense(64, activation="relu")(x)
    x_out = Dense(2, activation="softmax")(x)
    model = Model(inputs=[inx], outputs=[x_out])
    return model

batch = 256
tasa_aprend = 0.001
dropout = 0.1
bin_size=32

menor, medio, alto = 32,128,128

print('El lr = ', tasa_aprend)
print('El dropout = ', dropout)
print('El bin_size = ', bin_size)
print("batch size; ", batch)
print("kernel tamaños; ", menor, medio, alto)

El lr =  0.001
El dropout =  0.1
El bin_size =  32
batch size;  256
kernel tamaños;  32 128 128


In [ ]:
# Intento de reducir VRAM

#from tensorflow.keras.mixed_precision import set_global_policy
#set_global_policy('mixed_float16')

strategy = tf.distribute.MirroredStrategy()

#######################
    
# Datos a np.array
data = np.array(data, dtype=np.float32)  
labels = np.array(labels, dtype=np.float32)

# Generar una permutación de índices
indices = np.random.permutation(len(data))

# Mezclar los datos y etiquetas usando los índices
data = data[indices]
labels = labels[indices]

# Reservar 10%
val_size = int(0.1 * len(data))
x_final_val = data[:val_size]
y_final_val = labels[:val_size]

# 90% para el proceso
data = data[val_size:]
labels = labels[val_size:]

# Suponiendo que data y labels son tus datos y etiquetas
#kf = KFold(n_splits=5)  # Número de pliegues (folds)

# Lista para guardar los historiales de entrenamiento de cada fold
histories = []

#for train_index, val_index in kf.split(data):
#    X_train, X_val = data[train_index], data[val_index]
#    y_train, y_val = labels[train_index], labels[val_index]
   
# Dividir data
split_idx = int(len(data) * 0.8)
X_train, X_val = data[:split_idx], data[split_idx:]
y_train, y_val = labels[:split_idx], labels[split_idx:]
    
# Utilizar la estrategia distribuida para la creación y compilación del modelo
with strategy.scope():
    # Obtener modelo
    model = get_model(dp=dropout, n1=menor, n2=medio, n3=alto)
    adam = Adam(learning_rate=tasa_aprend)
    model.compile(optimizer=adam, loss='binary_crossentropy', 
                  metrics=['accuracy', tf.keras.metrics.AUC(), tf.keras.metrics.Precision(), 
                           tf.keras.metrics.Recall(), tf.keras.metrics.BinaryAccuracy()])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Entrenar el modelo
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), 
                    batch_size=batch, shuffle=True, epochs=30, callbacks=[early_stopping])

# Guardar el historial de entrenamiento
histories.append(history)

model.save('CNN CV Model')

Epoch 1/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 1654s 13s/step - accuracy: 0.8128 - auc: 0.8756 - binary_accuracy: 0.8128 - loss: 2.7358 - precision: 0.8128 - recall: 0.8128 - val_accuracy: 0.4902 - val_auc: 0.6865 - val_binary_accuracy: 0.4902 - val_loss: 1.0774 - val_precision: 0.4902 - val_recall: 0.4902
Epoch 2/30
130/130 ━━━━━━━━━━━━━━━━━━━━ 1634s 13s/step - accuracy: 0.8820 - auc: 0.9363 - binary_accuracy: 0.8820 - loss: 0.5146 - precision: 0.8820 - recall: 0.8820 - val_accuracy: 0.7367 - val_auc: 0.7689 - val_binary_accuracy: 0.7367 - val_loss: 0.6650 - val_precision: 0.7367 - val_recall: 0.7367


In [ ]:
model.save('/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Red Convolucional/CNN CV Model.keras')

In [ ]:
predicted = model.predict(x_final_val)
predicted